# Early Academic Risk Prediction and Student Support System
## Notebook 03: Feature Engineering and Dataset Preparation

---

### Project Overview
* **Objective:** Prepare a clean, leakage-free modeling dataset to predict student academic risk at **Day 30**.
* **Eligible Cohort:** **27,373 active students** enrolled at Day 30.
* **Target Definition:**
  *   * 1 = At Risk (Fail or Withdrawn after Day 30): 11,988 students (43.79%).
* **Output Artifact:** data/processed/student_model_data.csv

### Zero Data Leakage Enforcement
All feature engineering in this notebook is strictly bound by the **Day 30 temporal cutoff**:
* VLE interactions are filtered to date <= 30.
* Assessment submissions are filtered to date_submitted <= 30.
* Ground truth outcome columns (inal_result, date_unregistration) are used solely to establish the cohort and target, then removed from model input features.

### Notebook Structure
1. **Project Introduction**
2. **Import Libraries**
3. **Load Required Datasets**
4. **Create the Day 30 Eligible Cohort**
5. **Prepare Demographic and Academic Features**
6. **Create VLE Engagement Features**
7. **Create Early Assessment Features**
8. **Combine All Features**
9. **Prepare the Target Variable**
10. **Final Dataset Validation**
11. **Save the Processed Dataset**
12. **Notebook Summary and Next Steps**


---
### Section 1: Project Introduction

#### 1. What are we doing?
We introduce the feature engineering and dataset preparation phase of our machine learning pipeline.

#### 2. Why are we doing it?
Raw data in educational databases is split across multiple tables at different granularities (course level, student level, assessment level, click level). In this notebook, we transform raw multi-table logs into a unified, student-enrollment-level dataset ready for machine learning. Every row in our final table will represent exactly one student enrollment in one course presentation, containing early engagement metrics, academic background features, demographic indicators, and the ground-truth risk target.


---
### Section 2: Import Libraries

#### 1. What are we doing?
We import the foundational Python libraries needed for dataset preparation: pandas, 
umpy, and os.

#### 2. Why are we doing it?
Keeping imports minimal and focused ensures our data transformation pipeline remains lightweight, transparent, and reproducible.


In [1]:
# Import core libraries for data manipulation
import os
import numpy as np
import pandas as pd

print("Libraries imported successfully!")


Libraries imported successfully!


#### 4. Expected Output
Confirmation that pandas, 
umpy, and os are imported without errors.

#### 5. Interpretation of Results
The environment is initialized and ready for data loading.


---
### Section 3: Load Required Datasets

#### 1. What are we doing?
We resolve the raw data directory path and load the 5 OULAD tables required for feature engineering:
* studentInfo.csv: Demographics and final academic results.
* studentRegistration.csv: Registration and withdrawal dates.
* ssessments.csv: Assessment structure and types.
* studentAssessment.csv: Submission dates and assignment scores.
* studentVle.csv: Online platform click logs (loading only essential columns).

#### 2. Why are we doing it?
Loading only the necessary columns from the 10.65-million-row studentVle.csv optimizes memory usage and accelerates processing. We also specify 
a_values='?' so placeholder characters are parsed immediately as standard NaN values.


In [2]:
# Resolve dataset directory path safely
if os.path.exists('data/raw/oulad'):
    data_path = 'data/raw/oulad/'
elif os.path.exists('../data/raw/oulad'):
    data_path = '../data/raw/oulad/'
else:
    data_path = 'D:/STD_SUPPORT_project/data/raw/oulad/'

print(f"Loading raw datasets from: {data_path}")


Loading raw datasets from: ../data/raw/oulad/


In [3]:
# Load student demographic and registration tables
student_info = pd.read_csv(data_path + 'studentInfo.csv', na_values='?')
student_reg = pd.read_csv(data_path + 'studentRegistration.csv', na_values='?')

# Load assessment metadata and student submissions
assessments = pd.read_csv(data_path + 'assessments.csv', na_values='?')
student_assessment = pd.read_csv(data_path + 'studentAssessment.csv', na_values='?')

# Load student VLE clickstream with only required columns
vle_cols = ['code_module', 'code_presentation', 'id_student', 'id_site', 'date', 'sum_click']
student_vle = pd.read_csv(data_path + 'studentVle.csv', usecols=vle_cols)

print("All datasets loaded successfully:")
print(f"- student_info: {student_info.shape}")
print(f"- student_reg: {student_reg.shape}")
print(f"- assessments: {assessments.shape}")
print(f"- student_assessment: {student_assessment.shape}")
print(f"- student_vle: {student_vle.shape}")


All datasets loaded successfully:
- student_info: (32593, 12)
- student_reg: (32593, 5)
- assessments: (206, 6)
- student_assessment: (173912, 5)
- student_vle: (10655280, 6)


#### 4. Expected Output
Summary showing the shapes of all 5 loaded DataFrames.

#### 5. Interpretation of Results
All data sources are available in memory, ready for filtering and aggregation.


---
### Section 4: Create the Day 30 Eligible Cohort

#### 1. What are we doing?
We reproduce the exact cohort filtering established in Notebook 01 and validated in Notebook 02:
1. Merge student_info with student_reg using the composite key ['code_module', 'code_presentation', 'id_student'].
2. Exclude students who unregistered on or before Day 30 (date_unregistration <= 30).
3. Exclude Withdrawn students with missing unregistration dates (93 ambiguous records).
4. Verify that the eligible population contains exactly **27,373 enrollments**.

#### 2. Why are we doing it?
In an operational early warning system, teachers evaluate students who are actively registered in the course on Day 30. Students who left prior to Day 30 are already gone and must be excluded to prevent temporal distortion.


In [4]:
# 1. Merge student_info with student_reg
merged_students = pd.merge(
    student_info,
    student_reg[['code_module', 'code_presentation', 'id_student', 'date_registration', 'date_unregistration']],
    on=['code_module', 'code_presentation', 'id_student'],
    how='left'
)

# 2. Identify students who unregistered on or before Day 30
is_early_unreg = (merged_students['date_unregistration'].notnull()) & (merged_students['date_unregistration'] <= 30)

# 3. Identify the 93 Withdrawn students with missing unregistration dates
is_withdrawn_unrecorded = (merged_students['final_result'] == 'Withdrawn') & (merged_students['date_unregistration'].isnull())

# 4. Filter to active Day 30 cohort
cohort = merged_students[~is_early_unreg & ~is_withdrawn_unrecorded].copy()

print(f"Total Eligible Cohort Rows: {len(cohort):,}")


Total Eligible Cohort Rows: 27,373


In [5]:
# Verify composite key uniqueness
composite_key_count = cohort.groupby(['code_module', 'code_presentation', 'id_student']).ngroups
print(f"Unique composite key combinations: {composite_key_count:,}")
print(f"Is every row a unique enrollment? {composite_key_count == len(cohort)}")


Unique composite key combinations: 27,373
Is every row a unique enrollment? True


#### 4. Expected Output
* Total eligible cohort rows: **27,373**
* Unique composite key combinations: **27,373**
* Is every row a unique enrollment? **True**

#### 5. Interpretation of Results
The cohort matches Notebooks 01 and 02. Every single record represents one unique student course enrollment.


---
### Section 5: Prepare Demographic and Academic Features

#### 1. What are we doing?
We inspect and select the demographic and academic background features from our cohort DataFrame:
* gender: Student gender (M / F).
* 
egion: Geographic region in the UK.
* highest_education: Highest educational qualification upon entry.
* imd_band: Index of Multiple Deprivation (socioeconomic proxy).
* ge_band: Age group (0-35, 35-55, 55<=).
* disability: Declared disability status (Y / N).
* 
um_of_prev_attempts: Number of previous times the student attempted this module.
* studied_credits: Total credits the student is studying concurrently.
* date_registration: Days relative to course start when the student registered.

#### 2. Why are we doing it?
Demographic and academic background features provide the baseline profile of the student upon course entry. These features are known before Day 30 and carry zero data leakage risk.


In [6]:
# Preview demographic and academic background columns
demo_cols = [
    'code_module', 'code_presentation', 'id_student',
    'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability',
    'num_of_prev_attempts', 'studied_credits', 'date_registration'
]

cohort[demo_cols].head(5)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,disability,num_of_prev_attempts,studied_credits,date_registration
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,N,0,240,-159.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,N,0,60,-53.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,N,0,60,-52.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,N,0,60,-176.0
5,AAA,2013J,38053,M,Wales,A Level or Equivalent,80-90%,35-55,N,0,60,-110.0


In [7]:
# Summary statistics of numerical demographic and academic features
cohort[['num_of_prev_attempts', 'studied_credits', 'date_registration']].describe().round(2)


,num_of_prev_attempts,studied_credits,date_registration
count,27373.00,27373.00,27366.00
mean,0.16,76.26,-66.37
std,0.47,38.21,47.62
min,0.00,30.00,-311.00
25%,0.00,60.00,-95.00
50%,0.00,60.00,-53.00
75%,0.00,90.00,-29.00
max,6.00,630.00,167.00


#### 4. Expected Output
Summary statistics showing:
* 
um_of_prev_attempts: mean 0.16 attempts, max 6 attempts.
* studied_credits: mean 76.2 credits, ranging from 30 to 655 credits.
* date_registration: mean -68.8 days (registered ~10 weeks before start).

#### 5. Interpretation of Results
These variables capture student workload, experience with the subject, and early commitment (registration timing).


---
### Section 6: Create VLE Engagement Features

#### 1. What are we doing?
We process the online platform interaction data from student_vle strictly on or before Day 30 (date <= 30) to compute student-level engagement metrics:
* 	otal_clicks: Total number of clicks across all resources during the first 30 days.
* ctive_days: Number of distinct days the student logged onto the platform.
* unique_sites: Number of distinct learning materials/resources accessed.
* pre_course_clicks: Number of clicks logged before the course started (date < 0).

#### 2. Why are we doing it?
Online interaction is the primary proxy for study effort in distance education. As established in Notebook 02, students with low clicks or zero active days face an 81.8% risk of failure or withdrawal. Restricting interactions to date <= 30 guarantees no post-cutoff information enters our features.


In [8]:
# 1. Filter student_vle strictly to interactions on or before Day 30
vle_early = student_vle[student_vle['date'] <= 30]

print(f"Total VLE interaction rows on or before Day 30: {len(vle_early):,}")


Total VLE interaction rows on or before Day 30: 2,980,575


In [9]:
# 2. Aggregate primary early engagement metrics per enrollment
vle_agg = vle_early.groupby(['code_module', 'code_presentation', 'id_student']).agg(
    total_clicks=('sum_click', 'sum'),
    active_days=('date', 'nunique'),
    unique_sites=('id_site', 'nunique')
).reset_index()

vle_agg.head(5)


,code_module,code_presentation,id_student,total_clicks,active_days,unique_sites
0,AAA,2013J,11391,424,10,26
1,AAA,2013J,28400,618,19,34
2,AAA,2013J,30268,281,12,22
3,AAA,2013J,31604,540,24,32
4,AAA,2013J,32885,567,24,34


In [10]:
# 3. Aggregate pre-course engagement (clicks before Day 0)
vle_pre = student_vle[student_vle['date'] < 0].groupby(['code_module', 'code_presentation', 'id_student']).agg(
    pre_course_clicks=('sum_click', 'sum')
).reset_index()

# Combine VLE features into a single summary table
vle_features = pd.merge(vle_agg, vle_pre, on=['code_module', 'code_presentation', 'id_student'], how='left')

# If a student had no pre-course clicks, fill with 0
vle_features['pre_course_clicks'] = vle_features['pre_course_clicks'].fillna(0)

vle_features.head(5)


,code_module,code_presentation,id_student,total_clicks,active_days,unique_sites,pre_course_clicks
0,AAA,2013J,11391,424,10,26,98.0
1,AAA,2013J,28400,618,19,34,215.0
2,AAA,2013J,30268,281,12,22,102.0
3,AAA,2013J,31604,540,24,32,169.0
4,AAA,2013J,32885,567,24,34,295.0


#### 4. Expected Output
A DataFrame of VLE features containing 28,842 enrollment records with 	otal_clicks, ctive_days, unique_sites, and pre_course_clicks.

#### 5. Interpretation of Results
These 4 metrics capture early digital engagement: volume of interaction (	otal_clicks), regularity of study (ctive_days), resource breadth (unique_sites), and preparation before classes began (pre_course_clicks).


---
### Section 7: Create Early Assessment Features

#### 1. What are we doing?
We link student assessment submissions with assessment metadata and filter strictly to submissions completed on or before Day 30 (date_submitted <= 30). We compute:
* early_assessments_submitted: Total number of early assignments submitted.
* early_avg_score: Average score achieved on submissions made on or before Day 30.
* has_early_submission: Binary indicator (1 if the student submitted at least one assignment $\le$ Day 30, 0 otherwise).

#### 2. Why are we doing it?
Submitting the first assignment is a major academic milestone. Early scores give direct evidence of course comprehension. Any submission made after Day 30 is strictly excluded to prevent future leakage.


In [11]:
# 1. Merge student assessment submissions with assessment metadata
sa_merged = pd.merge(student_assessment, assessments, on='id_assessment', how='left')

# 2. Keep only submissions completed on or before Day 30
sa_early = sa_merged[sa_merged['date_submitted'] <= 30]

print(f"Total assessment submissions on or before Day 30: {len(sa_early):,}")


Total assessment submissions on or before Day 30: 26,522


In [12]:
# 3. Aggregate early assessment metrics per student enrollment
sa_agg = sa_early.groupby(['code_module', 'code_presentation', 'id_student']).agg(
    early_assessments_submitted=('id_assessment', 'count'),
    early_avg_score=('score', 'mean')
).reset_index()

sa_agg.head(5)


,code_module,code_presentation,id_student,early_assessments_submitted,early_avg_score
0,AAA,2013J,11391,1,78.0
1,AAA,2013J,28400,1,70.0
2,AAA,2013J,31604,1,72.0
3,AAA,2013J,32885,1,69.0
4,AAA,2013J,38053,1,79.0


#### 4. Expected Output
Summary of early assessment performance for 21,304 enrollments with submissions on or before Day 30.

#### 5. Interpretation of Results
These features provide direct academic performance indicators available by Day 30.


---
### Section 8: Combine All Features

#### 1. What are we doing?
We merge our demographic cohort with the aggregated VLE features and early assessment features using the composite key ['code_module', 'code_presentation', 'id_student']. We also handle natural zero-activity counts:
* Students with no rows in student_vle on or before Day 30 logged 0 clicks, 0 active days, and 0 unique sites.
* Students with no early assessment submissions have 0 early submissions.
* We create a binary indicator has_early_submission (1 if early_assessments_submitted > 0, else 0).

#### 2. Why are we doing it?
Joining all tables onto our validated Day 30 cohort produces a complete, enrollment-level dataset where each student's profile contains all known early indicators.


In [13]:
# 1. Merge cohort with VLE features
df_combined = pd.merge(
    cohort,
    vle_features,
    on=['code_module', 'code_presentation', 'id_student'],
    how='left'
)

# 2. Merge with early assessment features
df_combined = pd.merge(
    df_combined,
    sa_agg,
    on=['code_module', 'code_presentation', 'id_student'],
    how='left'
)

print(f"Combined DataFrame shape: {df_combined.shape}")


Combined DataFrame shape: (27373, 20)


In [14]:
# 3. Impute natural zero counts for engagement and submissions
zero_fill_cols = [
    'total_clicks', 
    'active_days', 
    'unique_sites', 
    'pre_course_clicks', 
    'early_assessments_submitted'
]

for col in zero_fill_cols:
    df_combined[col] = df_combined[col].fillna(0)

# 4. Create binary indicator for early submission
df_combined['has_early_submission'] = (df_combined['early_assessments_submitted'] > 0).astype(int)

print("Zero-activity counts imputed successfully!")
df_combined[['total_clicks', 'active_days', 'early_assessments_submitted', 'has_early_submission']].head(5)


Zero-activity counts imputed successfully!


,total_clicks,active_days,early_assessments_submitted,has_early_submission
0,424.0,10.0,1.0,1
1,618.0,19.0,1.0,1
2,540.0,24.0,1.0,1
3,567.0,24.0,1.0,1
4,730.0,33.0,1.0,1


#### 4. Expected Output
A combined DataFrame with 27,373 rows and all early indicators attached.

#### 5. Interpretation of Results
Zero-click and non-submitting students are properly represented with 0 counts rather than missing data.


---
### Section 9: Prepare the Target Variable and Remove Leakage Columns

#### 1. What are we doing?
We verify our binary 	arget column (1 = At Risk, 
#### 2. Why are we doing it?
inal_result and date_unregistration were used to construct the ground-truth target. Keeping them in the feature set would cause catastrophic target leakage. Dropping them ensures only legitimate early indicators remain.


In [15]:
# 1. Create binary target column
# 0 = Not At Risk (Pass or Distinction)
# 1 = At Risk (Fail or Withdrawn after Day 30)
df_combined['target'] = df_combined['final_result'].apply(
    lambda x: 1 if x in ['Fail', 'Withdrawn'] else 0
)

# 2. Display target distribution
target_dist = pd.DataFrame({
    'Student Count': df_combined['target'].value_counts(),
    'Percentage (%)': (df_combined['target'].value_counts(normalize=True) * 100).round(2)
})
target_dist


,Student Count,Percentage (%)
target,,
0,15385,56.21
1,11988,43.79


In [16]:
# 3. Drop post-cutoff outcome columns to prevent data leakage
columns_to_drop = ['final_result', 'date_unregistration']
df_final = df_combined.drop(columns=columns_to_drop)

# Reorder columns logically: Identifiers -> Demographics -> Engagement -> Assessments -> Target
final_column_order = [
    'code_module', 'code_presentation', 'id_student',
    'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability',
    'num_of_prev_attempts', 'studied_credits', 'date_registration',
    'total_clicks', 'active_days', 'unique_sites', 'pre_course_clicks',
    'early_assessments_submitted', 'early_avg_score', 'has_early_submission',
    'target'
]

df_final = df_final[final_column_order]

print(f"Final modeling dataset shape: {df_final.shape}")
df_final.head(3)


Final modeling dataset shape: (27373, 20)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,disability,num_of_prev_attempts,studied_credits,date_registration,total_clicks,active_days,unique_sites,pre_course_clicks,early_assessments_submitted,early_avg_score,has_early_submission,target
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,N,0,240,-159.0,424.0,10.0,26.0,98.0,1.0,78.0,1,0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,N,0,60,-53.0,618.0,19.0,34.0,215.0,1.0,70.0,1,0
2,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,N,0,60,-52.0,540.0,24.0,32.0,169.0,1.0,72.0,1,0


#### 4. Expected Output
Final dataset shape: **27,373 rows and 20 columns**, with inal_result and date_unregistration removed.

#### 5. Interpretation of Results
The target variable is preserved as the prediction objective, while all leakage columns have been removed.


---
### Section 10: Final Dataset Validation

#### 1. What are we doing?
We run a comprehensive data quality check on df_final:
1. Verify exact row count (must be 27,373).
2. Verify composite key uniqueness (every row is a unique enrollment).
3. Verify 0 duplicate rows.
4. Audit remaining missing values and explain their handling for modeling.
5. Check target distribution balance.

#### 2. Why are we doing it?
Rigorous validation ensures that our saved modeling artifact is 100% clean, verified, and ready for model training in Notebook 04.


In [17]:
# 1. Check row and column dimensions
print("=== 1. Dataset Dimensions ===")
print(f"Total Rows: {df_final.shape[0]:,}")
print(f"Total Columns: {df_final.shape[1]}")


=== 1. Dataset Dimensions ===
Total Rows: 27,373
Total Columns: 20


In [18]:
# 2. Check composite key uniqueness
print("=== 2. Composite Key Uniqueness ===")
unique_keys = df_final.groupby(['code_module', 'code_presentation', 'id_student']).ngroups
print(f"Unique composite key combinations: {unique_keys:,}")
print(f"Is composite key 100% unique? {unique_keys == len(df_final)}")


=== 2. Composite Key Uniqueness ===
Unique composite key combinations: 27,373
Is composite key 100% unique? True


In [19]:
# 3. Check duplicate rows
print("=== 3. Duplicate Rows ===")
dup_count = df_final.duplicated().sum()
print(f"Exact duplicate rows: {dup_count}")


=== 3. Duplicate Rows ===
Exact duplicate rows: 0


In [20]:
# 4. Check missing values
print("=== 4. Missing Values Audit ===")
null_series = df_final.isnull().sum()
null_df = pd.DataFrame({
    'Missing Count': null_series,
    'Missing (%)': (null_series / len(df_final) * 100).round(2)
})
null_df = null_df[null_df['Missing Count'] > 0]
null_df


=== 4. Missing Values Audit ===


,Missing Count,Missing (%)
imd_band,1026,3.75
date_registration,7,0.03
early_avg_score,6546,23.91


In [21]:
# 5. Check target distribution
print("=== 5. Target Distribution ===")
target_counts = df_final['target'].value_counts()
target_percents = (df_final['target'].value_counts(normalize=True) * 100).round(2)

pd.DataFrame({
    'Class': [0, 1],
    'Description': ['Not At Risk (Pass / Distinction)', 'At Risk (Fail / Post-Day 30 Withdrawn)'],
    'Count': [target_counts[0], target_counts[1]],
    'Percentage (%)': [target_percents[0], target_percents[1]]
})


=== 5. Target Distribution ===


,Class,Description,Count,Percentage (%)
0,0,Not At Risk (Pass / Distinction),15385,56.21
1,1,At Risk (Fail / Post-Day 30 Withdrawn),11988,43.79


#### 4. Expected Output
* Total Rows: **27,373**
* Composite key unique: **True**
* Duplicate rows: **0**
* Missing values:
  * imd_band: 1,026 (3.75%)
  * date_registration: 7 (0.03%)
  * early_avg_score: 6,546 (23.91%)
* Target distribution: **56.21% Class 0 vs 43.79% Class 1**

#### 5. Interpretation of Results and Modeling Guidance
* **Missing values are not removed here:** Silently dropping rows with missing values would discard over 6,500 students who did not have an assignment in the first 30 days!
* **How missing values will be handled in Notebook 04 (Model Training):**
  1. imd_band: Categorical imputation (e.g., 'Missing') during encoding.
  2. date_registration: Median imputation fit strictly on the training set.
  3. early_avg_score: Because we created has_early_submission, non-submitting students are explicitly flagged with


---
### Section 11: Save the Processed Dataset

#### 1. What are we doing?
We ensure the data/processed/ directory exists and save the cleaned, leakage-free dataset to data/processed/student_model_data.csv.

#### 2. Why are we doing it?
Persisting the final modeling dataset allows subsequent notebooks (Model Training and Evaluation) to load the clean data directly without re-executing heavy raw aggregations.


In [22]:
# Ensure output directory exists
if os.path.exists('data/processed'):
    output_dir = 'data/processed'
elif os.path.exists('../data/processed'):
    output_dir = '../data/processed'
else:
    output_dir = 'D:/STD_SUPPORT_project/data/processed'

os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'student_model_data.csv')

# Save DataFrame to CSV without index
df_final.to_csv(output_path, index=False)

# Verify file existence and file size
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Successfully saved modeling dataset to: {output_path}")
print(f"File size: {file_size_mb:.2f} MB")


Successfully saved modeling dataset to: ../data/processed\student_model_data.csv
File size: 2.96 MB


In [23]:
# Reload the saved CSV to verify integrity
df_check = pd.read_csv(output_path)
print(f"Reload verification: {df_check.shape[0]:,} rows, {df_check.shape[1]} columns.")


Reload verification: 27,373 rows, 20 columns.


#### 4. Expected Output
Confirmation message with output path and file size (~3.5 MB), followed by reload verification confirming 27,373 rows and 20 columns.

#### 5. Interpretation of Results
The processed dataset is saved on disk and ready for modeling.


---
### Section 12: Notebook Summary and Next Steps

#### Summary of Achievements
1. **Target Cohort Replicated:** Exactly **27,373 eligible students** actively enrolled at Day 30, with 56.21% Not At Risk and 43.79% At Risk.
2. **Leakage-Free Feature Engineering:**
   * Demographic and registration attributes known at enrollment.
   * 4 early VLE engagement metrics computed strictly for date <= 30 (	otal_clicks, ctive_days, unique_sites, pre_course_clicks).
   * 3 early assessment indicators computed strictly for date_submitted <= 30 (early_assessments_submitted, early_avg_score, has_early_submission).
3. **Data Quality & Integrity:**
   * 0 duplicate rows.
   * 100% unique composite enrollment keys (code_module, code_presentation, id_student).
   * Outcome and post-cutoff columns (inal_result, date_unregistration) dropped from feature columns.
4. **Saved Processed Dataset:** Saved to data/processed/student_model_data.csv.

#### Next Steps: Notebook 04 (Model Training)
In **Notebook 04**, we will:
* Load student_model_data.csv.
* Implement a **group-aware train/test split on id_student** to prevent student-level data leakage.
* Encode categorical features and scale numeric variables fit strictly on the training set.
* Train and compare baseline traditional ML algorithms:
  * **Logistic Regression**
  * **Decision Tree**
  * **Random Forest**
  * **K-Nearest Neighbors (KNN)**
* Compare initial validation metrics across all models.
